**06 · Verificación de resultados y checklist de cierre**

Ejecutar desde la raíz de `point-field`. Este cuaderno no reemplaza la revisión manual, pero automatiza la verificación de archivos, tablas y cifras clave antes de la última pulida del documento.

In [2]:
from pathlib import Path
import sys, subprocess, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'benchmarks'/'common.py').exists() and (p/'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError("Abre Jupyter desde la raíz de point-field o una subcarpeta del repositorio.")
sys.path.insert(0, str(ROOT/'benchmarks'))
OUT = ROOT/'runs'/'tesis_figuras'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'figure.dpi': 120, 'savefig.dpi': 300, 'pdf.fonttype': 42})
COLORS = ['#21618c','#c65036','#63933e','#8b65a5','#c49a38','#458e91','#d16c9f','#555555']
def save(fig, name):
    for ext in ('pdf','png'):
        fig.savefig(OUT/f'{name}.{ext}', bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)
def points(ax, X, y=None, s=6):
    X=np.asarray(X)
    if y is None:
        if X.shape[1]==2: ax.scatter(X[:,0],X[:,1],s=s,c=COLORS[0],linewidths=0)
        else: ax.scatter(X[:,0],X[:,1],X[:,2],s=s,c=COLORS[0],linewidths=0)
    else:
        labs=np.unique(y)
        for j,c in enumerate(labs):
            col='#b9b9b9' if c==-1 else COLORS[j%len(COLORS)]
            m=(y==c)
            if X.shape[1]==2: ax.scatter(X[m,0],X[m,1],s=s,c=col,linewidths=0)
            else: ax.scatter(X[m,0],X[m,1],X[m,2],s=s,c=col,linewidths=0)
    ax.set_xticks([]); ax.set_yticks([])
    if X.shape[1]==2:
        ax.set_aspect('equal', adjustable='box')
    else:
        ax.set_zticks([]); ax.set_box_aspect((1,1,1)); ax.view_init(elev=23, azim=35)
def run_script(name, *args):
    subprocess.run([sys.executable, str(ROOT/'benchmarks'/name), *map(str,args)], cwd=ROOT, check=True)
print('Repositorio:', ROOT, 'Salidas:', OUT)
try:
    commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
except Exception:
    commit='No disponible'
print('Commit:', commit)


Repositorio: /home/icantu24/Documents/PEF-CD/point-field Salidas: /home/icantu24/Documents/PEF-CD/point-field/runs/tesis_figuras
Commit: 88445aa61ca2a66afcc53a931cf0927232ef232f


**1. Manifesto mínimo de archivos.** Ajusta la lista si tus archivos históricos tienen nombres diferentes. El objetivo es saber qué existe, qué falta y qué se va a volver a generar.

In [7]:
EXPECTED = {
    "clustering_2d": [
        ROOT / "benchmarks/results/clustering_2d_eval.csv"
    ],
    "objetos_sinteticos": [
        ROOT / "benchmarks/results/objetos_3d.csv"
    ],
    "modelnet10": [
        ROOT / "benchmarks/results/modelnet10_test.csv"
    ],
    "modelnet40": [
        ROOT / "benchmarks/results/modelnet40_test.csv"
    ],
    "embeddings": [
        ROOT / "benchmarks/results/embeddings_2d_eval.csv"
    ],
    "contacto3d": [
        ROOT / "runs/20261119T210929036674Z/scenes_3d.csv"
    ],
    "flare": [
        ROOT / "benchmarks/results/aplicacion_FLARE_nivel2.csv"
    ],
    "reconstruccion": [
        ROOT / "runs/20261119T210929036674Z/reconstruction_per_object.csv"
    ],
    "grado_puntos": [
        ROOT / "runs/tesis_figuras/09_metricas_soporte_annulus.csv"
    ],
    "diagnostico_numerico": [
        ROOT / "runs/tesis_figuras/09_diagnostico_annulus.csv"
    ]
}

rows = []
loaded = {}

for key, candidates in EXPECTED.items():

    found = next(
        (p for p in candidates if p.exists()),
        None
    )

    rows.append({
        "bloque": key,
        "archivo": str(found) if found else "NO ENCONTRADO",
        "estado": "OK" if found else "FALTA"
    })

    if found is not None:
        loaded[key] = pd.read_csv(found)

manifest = pd.DataFrame(rows)

display(manifest)

manifest.to_csv(
    OUT / "06_manifest_resultados.csv",
    index=False
)


,bloque,archivo,estado
0,clustering_2d,/home/icantu24/Documents/PEF-CD/point-field/be...,OK
1,objetos_sinteticos,/home/icantu24/Documents/PEF-CD/point-field/be...,OK
2,modelnet10,/home/icantu24/Documents/PEF-CD/point-field/be...,OK
3,modelnet40,/home/icantu24/Documents/PEF-CD/point-field/be...,OK
4,embeddings,/home/icantu24/Documents/PEF-CD/point-field/be...,OK
5,contacto3d,/home/icantu24/Documents/PEF-CD/point-field/ru...,OK
6,flare,/home/icantu24/Documents/PEF-CD/point-field/be...,OK
7,reconstruccion,/home/icantu24/Documents/PEF-CD/point-field/ru...,OK
8,grado_puntos,/home/icantu24/Documents/PEF-CD/point-field/ru...,OK
9,diagnostico_numerico,/home/icantu24/Documents/PEF-CD/point-field/ru...,OK


**2. Resúmenes automáticos.** Si un CSV existe, se resume con medias, desviaciones y tablas pivote. Esto te permite cotejar si las cifras del manuscrito coinciden con lo que realmente está en disco.

In [8]:
def first_existing(paths):
    return next((p for p in paths if p.exists()), None)

def summarize_csv(path):
    df=pd.read_csv(path)
    print('Archivo:', path)
    print('Columnas:', list(df.columns))
    display(df.head())
    numeric=df.select_dtypes(include='number')
    if not numeric.empty:
        display(numeric.describe().T)
    return df

loaded={}
for key,paths in EXPECTED.items():
    path=first_existing(paths)
    if path:
        try:
            loaded[key]=summarize_csv(path)
        except Exception as e:
            print('No se pudo resumir', key, e)


Archivo: /home/icantu24/Documents/PEF-CD/point-field/benchmarks/results/clustering_2d_eval.csv
Columnas: ['case', 'held_out', 'variant', 'seed', 'method', 'family', 'receives_k', 'n_points', 'ARI', 'NMI', 'n_found', 'noise_frac', 'time_s', 'szego_active', 'winding_gain', 'dbscan_eps']


,case,held_out,variant,seed,method,family,receives_k,n_points,ARI,NMI,n_found,noise_frac,time_s,szego_active,winding_gain,dbscan_eps
0,compacto,False,base,100,K-Means,referencia,True,450,0.903218,0.869259,3,0.000000,0.801771,NaN,NaN,NaN
1,compacto,False,base,100,DBSCAN,referencia,False,450,0.858343,0.809593,3,0.060000,0.003726,NaN,NaN,0.16747
2,compacto,False,base,100,OPTICS,referencia,False,450,0.487982,0.579457,3,0.355556,0.252167,NaN,NaN,NaN
3,compacto,False,base,100,Spectral (kNN),referencia,True,450,0.903458,0.875769,3,0.000000,0.636780,NaN,NaN,NaN
4,compacto,False,base,100,CD global (inicial),CD,True,450,0.915759,0.887091,3,0.000000,0.786937,NaN,NaN,NaN


,count,mean,std,min,25%,50%,75%,max
seed,840.0,102.000000,1.415056,100.000000,101.000000,102.000000,103.000000,104.000000
n_points,840.0,291.783333,86.959120,155.000000,225.000000,300.000000,309.250000,450.000000
ARI,840.0,0.422130,0.378854,-0.007366,0.096837,0.253812,0.903659,1.000000
NMI,840.0,0.469153,0.351681,0.000000,0.149595,0.395008,0.864025,1.000000
n_found,840.0,3.548810,3.091377,1.000000,2.000000,3.000000,3.000000,22.000000
noise_frac,840.0,0.064924,0.150143,0.000000,0.000000,0.000000,0.027827,0.791111
time_s,840.0,0.285825,0.577636,0.001858,0.008243,0.033126,0.218630,4.610847
winding_gain,120.0,0.133996,0.108408,0.000285,0.038512,0.108565,0.206081,0.436892
dbscan_eps,120.0,0.171553,0.039511,0.103138,0.145755,0.164706,0.183691,0.294646


Archivo: /home/icantu24/Documents/PEF-CD/point-field/benchmarks/results/objetos_3d.csv
Columnas: ['condition', 'method', 'mAP', 'nn', 'ci_low', 'ci_high']


,condition,method,mAP,nn,ci_low,ci_high
0,clean,CD_U,0.912464,1.000000,NaN,NaN
1,clean,CD_W,0.915957,1.000000,NaN,NaN
2,clean,M_U,0.840234,1.000000,NaN,NaN
3,clean,D2,0.758929,0.957143,NaN,NaN
4,clean,CD_U_metric,0.939663,1.000000,NaN,NaN


,count,mean,std,min,25%,50%,75%,max
mAP,40.0,0.554717,0.346660,-0.293310,0.330966,0.663490,0.870140,0.939663
nn,30.0,0.796667,0.218100,0.342857,0.592857,0.892857,1.000000,1.000000
ci_low,10.0,0.006484,0.127961,-0.332304,0.001693,0.032548,0.037477,0.149623
ci_high,10.0,0.052796,0.130019,-0.253674,0.018927,0.067023,0.078425,0.217133


... (20 outputs left out)

,count,mean,std,min,25%,50%,75%,max
seed,64.0,100.000000,0.000000,100.000000,100.000000,100.000000,100.000000,100.000000
degree,64.0,10.125000,9.594145,1.000000,3.500000,7.000000,13.000000,32.000000
n_points,64.0,3975.000000,5223.816035,100.000000,425.000000,1500.000000,5000.000000,16000.000000
basis_size,64.0,112.750000,177.384704,3.000000,12.750000,36.500000,106.500000,561.000000
threshold,64.0,119.071154,117.945204,4.046716,26.379780,82.165373,189.850522,428.262768
fit_time_s,64.0,0.031289,0.088943,0.000170,0.000358,0.001623,0.008697,0.584526
eval_time_s,64.0,0.244303,0.415885,0.001711,0.056992,0.089499,0.182576,1.745564
IoU,64.0,0.790187,0.172516,0.070100,0.711171,0.835078,0.913388,0.932611
Precision,64.0,0.894527,0.101836,0.721302,0.814527,0.954112,0.975424,1.000000
Recall,64.0,0.883940,0.177930,0.070100,0.920386,0.946686,0.948832,0.952122


Archivo: /home/icantu24/Documents/PEF-CD/point-field/runs/tesis_figuras/09_diagnostico_annulus.csv
Columnas: ['degree', 'n_points', 'basis_size', 'N_per_basis', 'necessarily_rank_deficient', 'cond_M', 'cond_regularized', 'effective_dof', 'regularization', 'threshold', 'fit_time_s', 'eval_time_s', 'status']


,degree,n_points,basis_size,N_per_basis,necessarily_rank_deficient,cond_M,cond_regularized,effective_dof,regularization,threshold,fit_time_s,eval_time_s,status
0,1,100,3,33.333333,False,3.442427,3.442427,3.0,5.410476e-09,4.096819,0.000264,0.015141,OK
1,1,200,3,66.666667,False,3.430420,3.430420,3.0,5.413771e-09,4.099963,0.000209,0.006650,OK
2,1,500,3,166.666667,False,3.427358,3.427358,3.0,5.426304e-09,4.118666,0.000197,0.001014,OK
3,1,1000,3,333.333333,False,3.232177,3.232177,3.0,5.427628e-09,4.054230,0.000203,0.001192,OK
4,1,2000,3,666.666667,False,3.220592,3.220592,3.0,5.447189e-09,4.046716,0.000218,0.009635,OK


,count,mean,std,min,25%,50%,75%,max
degree,64.0,1.012500e+01,9.594145e+00,1.000000e+00,3.500000e+00,7.000000e+00,1.300000e+01,3.200000e+01
n_points,64.0,3.975000e+03,5.223816e+03,1.000000e+02,4.250000e+02,1.500000e+03,5.000000e+03,1.600000e+04
basis_size,64.0,1.127500e+02,1.773847e+02,3.000000e+00,1.275000e+01,3.650000e+01,1.065000e+02,5.610000e+02
N_per_basis,64.0,3.199431e+02,8.264396e+02,1.782531e-01,7.014260e+00,3.452381e+01,1.763126e+02,5.333333e+03
cond_M,64.0,6.067872e+26,3.521220e+27,3.210499e+00,5.015463e+03,6.764238e+07,2.464524e+16,2.757220e+28
cond_regularized,64.0,4.121808e+09,7.312273e+09,3.210499e+00,5.015420e+03,6.360314e+07,4.626074e+09,2.268786e+10
effective_dof,64.0,5.086492e+01,4.545564e+01,3.000000e+00,1.274999e+01,3.641777e+01,8.097431e+01,1.449762e+02
regularization,64.0,1.575241e-09,1.781494e-09,5.861078e-11,2.982619e-10,7.955898e-10,2.075051e-09,5.447189e-09
threshold,64.0,1.190712e+02,1.179452e+02,4.046716e+00,2.637978e+01,8.216537e+01,1.898505e+02,4.282628e+02
fit_time_s,64.0,2.820259e-02,6.686012e-02,1.960600e-04,4.251518e-04,1.566460e-03,9.438766e-03,3.734753e-01


**3. Verificación de cifras clave reclamadas en la tesis.** Los alias y los nombres de columna pueden variar entre versiones; ajusta el diccionario si tu CSV usa nombres diferentes. El resultado es una tabla PASS/WARN con tolerancia configurable.

In [9]:
CLAIMS = [
    # ModelNet10

    ("modelnet10", "clean", "CD_U", 0.383049491),
    ("modelnet10", "clean", "CD_W", 0.365491617),
    ("modelnet10", "clean", "M_U", 0.398547161),
    ("modelnet10", "clean", "D2", 0.272120862),
    ("modelnet10", "clean", "CD_U_metric", 0.399551629),
    ("modelnet10", "clean", "CD_W_metric", 0.406358667),
    ("modelnet10", "clean", "M_U_metric", 0.406102127),

    # ModelNet40

    ("modelnet40", "clean", "CD_U", 0.224984093),
    ("modelnet40", "clean", "CD_W", 0.215241477),
    ("modelnet40", "clean", "M_U", 0.220037445),
    ("modelnet40", "clean", "D2", 0.151321009),
    ("modelnet40", "clean", "CD_U_metric", 0.244541788),
    ("modelnet40", "clean", "CD_W_metric", 0.246382275),
    ("modelnet40", "clean", "M_U_metric", 0.236354119),

    # FLARE

    ("flare", "all", "CD_U+m", 0.326148069),
    ("flare", "all", "M_U+m", 0.282097125),
    ("flare", "all", "CD_U", 0.259287861),
    ("flare", "all", "M_U", 0.263623100),
    ("flare", "all", "D2", 0.201064918)
]

TOL = 1e-6
checks = []

for dataset, condition, method, expected in CLAIMS:

    observed = np.nan

    if dataset not in loaded:

        status = "MISSING_FILE"

    else:

        data = loaded[dataset]

        if dataset == "flare":

            sub = data[
                data["descriptor"] == method
            ]

        else:

            sub = data[
                (data["method"] == method)
                & (data["condition"] == condition)
            ]

        if len(sub) == 0:

            status = "NO_MATCH"

        elif len(sub) > 1:

            status = "DUPLICATE"

        else:

            observed = float(sub["mAP"].iloc[0])

            status = (
                "PASS"
                if np.isclose(
                    observed,
                    expected,
                    atol=TOL,
                    rtol=0
                )
                else "FAIL"
            )

    checks.append({
        "dataset": dataset,
        "condition": condition,
        "method": method,
        "expected": expected,
        "observed": observed,
        "absolute_error": abs(observed - expected),
        "status": status
    })

check_df = pd.DataFrame(checks)

display(check_df)

check_df.to_csv(
    OUT / "06_checks_tesis.csv",
    index=False
)

print("\nResumen:")
print(check_df["status"].value_counts())

assert check_df["status"].eq("PASS").all(), (
    "Existen resultados que necesitan revisión."
)

,dataset,condition,method,expected,observed,absolute_error,status
0,modelnet10,clean,CD_U,0.383049,0.383049,4.328382e-10,PASS
1,modelnet10,clean,CD_W,0.365492,0.365492,3.068290e-10,PASS
2,modelnet10,clean,M_U,0.398547,0.398547,2.347704e-10,PASS
3,modelnet10,clean,D2,0.272121,0.272121,3.202201e-10,PASS
4,modelnet10,clean,CD_U_metric,0.399552,0.399552,1.939552e-10,PASS
5,modelnet10,clean,CD_W_metric,0.406359,0.406359,1.473354e-10,PASS
6,modelnet10,clean,M_U_metric,0.406102,0.406102,3.162165e-11,PASS
7,modelnet40,clean,CD_U,0.224984,0.224984,4.451245e-10,PASS
8,modelnet40,clean,CD_W,0.215241,0.215241,4.948345e-10,PASS
9,modelnet40,clean,M_U,0.220037,0.220037,4.940522e-10,PASS



Resumen:
status
PASS    19
Name: count, dtype: int64


**4. Checklist manual final.** Esta tabla se llena automáticamente con los bloques principales para que tú marques qué ya quedó validado, qué se va a volver a correr y qué quedará como limitación declarada.

In [10]:
checklist = pd.DataFrame([
    ['Soporte 2D','Figura de grado y puntos',''],
    ['Soporte 3D','Cobertura y franja próxima',''],
    ['Clustering 2D','Panel V0–V2, heatmaps y robustez',''],
    ['Clustering 3D','Escenas separadas y en contacto',''],
    ['Embeddings','Trustworthiness, continuity, 5-NN',''],
    ['Objetos sintéticos','mAP, 1-NN, invariancia/robustez',''],
    ['ModelNet10','mAP, 1-NN, comparación DNN',''],
    ['ModelNet40','mAP, 1-NN, comparación DNN',''],
    ['FLARE','mAP, 1-NN, alcance exploratorio',''],
    ['Documento','Citas, tablas, figuras y números consistentes',''],
], columns=['bloque','qué revisar','estado_manual'])
display(checklist)
checklist.to_csv(OUT/'06_checklist_manual.csv', index=False)
print('Guardados: 06_manifest_resultados.csv, 06_checks_tesis.csv y 06_checklist_manual.csv')


,bloque,qué revisar,estado_manual
0,Soporte 2D,Figura de grado y puntos,
1,Soporte 3D,Cobertura y franja próxima,
2,Clustering 2D,"Panel V0–V2, heatmaps y robustez",
3,Clustering 3D,Escenas separadas y en contacto,
4,Embeddings,"Trustworthiness, continuity, 5-NN",
5,Objetos sintéticos,"mAP, 1-NN, invariancia/robustez",
6,ModelNet10,"mAP, 1-NN, comparación DNN",
7,ModelNet40,"mAP, 1-NN, comparación DNN",
8,FLARE,"mAP, 1-NN, alcance exploratorio",
9,Documento,"Citas, tablas, figuras y números consistentes",


Guardados: 06_manifest_resultados.csv, 06_checks_tesis.csv y 06_checklist_manual.csv
